# Homework 8 -- ATOC 4815/5815

**Due: Thursday, 11/12, 11:59 pm (available until Friday, 11/13).**

This homework has two parts. Part A [60 pts] is a partially-completed notebook, like HW6/HW7: some
cells are already filled in for you, and some are marked **`# TODO`**. Part B [15 pts] is your first
progress submission toward the course project.

**Skills practiced (Part A):**
- Taking the exact same PCA pipeline we built today for *2 spatial dimensions* (`village.py` -- house
  locations in (East-West, North-South)) and applying it, unchanged, to *3 spectral dimensions*
  (a satellite image's Red, Green, Blue channels)
- Reprojecting data onto the eigenvectors to get **scores** -- today we only reprojected in class
  without naming the concept; here you'll compute and interpret them explicitly
- Reading a PCA "scores map" the way a remote-sensing paper would (same type of figure as the
  Shea, Pilewskie & Kindel (2011) vegetation-PC slide from today's lecture)

**Skills practiced (Part B):** reading your own project dataset with Python, no black-box tools,
and producing your first baseline plot -- see Part B below.


## Context: what this builds on

In today's lecture (`village.py`) we did PCA on a village of houses, each described by two numbers:
its East-West position and its North-South position. We:

1. Stacked all the houses' (EW, NS) pairs into one array, `A` (shape: `n_houses x 2`).
2. Found the **center** (mean) and the **covariance matrix** of `A`.
3. Did an eigenvalue decomposition of the covariance matrix: the eigenvectors are the "self"
   (intrinsic) directions of the point cloud -- the direction along the road, and the direction
   across it -- and the eigenvalues say how much spread (variance) is along each direction.
4. **Reprojected** every house's (EW, NS) location onto the two eigenvectors, using
   `A.dot(matrix_w)`. This gave each house two *new* numbers -- its coordinates in the eigenvector
   system instead of the (EW, NS) system.

Those new numbers from step 4 -- *how far each sample sits along each eigenvector* -- are called
**scores** (sometimes "PC scores" or "loadings on the component," though strictly "loading" refers
to the eigenvector itself and "score" to the projected value). We did this projection in class
today but didn't give it that name, and you haven't practiced it yet -- that's much of what this
homework does.

**Today's translation: spatial -> spectral.** Nothing about the PCA machinery above is specific to
"East-West" and "North-South." Those were just the two *features* describing each house. In this
homework, the *samples* are pixels of a satellite image (instead of houses), and the *features* are
the pixel's Red, Green, and Blue channel values (instead of EW, NS) -- three "spectral" channels
instead of two spatial ones. Everything else -- center, covariance matrix, eigenvectors, eigenvalues,
reprojection -- is the identical pipeline, just with `n_pixels x 3` instead of `n_houses x 2`. If you
understood `village.py`, you already understand the method here; the only new ideas are (a) what a
"score" is and (b) that reshaping the scores back into the image's rows/columns turns them into a
*map* -- showing *where* each pattern is strongest, exactly like the vegetation-PC figure from class.


### Where to put this notebook

Same convention as HW6/HW7:

1. `cd` into your `ATOC-4815` folder and run `git pull` -- that gets you `hw8.ipynb` in `py/`.
2. Download **`channel.png`** and put it in your `dat/` folder, next to `py/`, not part of the
   GitHub repo -- same as `helene.png` was for HW7.

```
ATOC-4815/
├── dat/
│   └── channel.png
└── py/
    └── hw8.ipynb   <-- you are here
```


In [ ]:
# --- Provided: load the image ---
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np

wesn = [-2.0077, 2.9159, 48.5883, 53.4864]   # [west, east, south, north] -- not critical for the PCA, just for display
img = mpimg.imread('../dat/channel.png')
print('image shape:', img.shape)   # (rows, cols, channels)

plt.figure(figsize=(7, 7))
plt.imshow(img, extent=wesn)
plt.xlabel('Longitude [deg]'); plt.ylabel('Latitude [deg]')
plt.title('Our satellite image -- same one referenced in today\'s lecture')
plt.show()


## Task 1 [10 pts] -- Finding scene types by eye

Look at the image above. There's ocean, clouds, and (if you look closely) vegetation on land.
The cell below already marks out a small rectangular region of **ocean** and one of **cloud**,
by pixel index (not lon/lat) -- the same "pick a box, read out its pixel indices" approach as
HW6's gridding exercise.

**TODO:** add a third box, `xv`/`yv`, around a patch of **vegetation** (anywhere you see a
greenish patch of land), and overplot it in green, the same way the other two are plotted.


In [ ]:
# --- Provided: ocean and cloud regions ---
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111)
ax.imshow(img)

xl = [600, 630]; yl = [720, 740]   # region 1 (ocean)
ax.plot(xl, [yl[0], yl[0]], 'w-', label='ocean'); ax.plot(xl, [yl[1], yl[1]], 'w-')
ax.plot([xl[0], xl[0]], yl, 'w-'); ax.plot([xl[1], xl[1]], yl, 'w-')

xc = [831, 900]; yc = [302, 380]   # region 2 (cloud)
ax.plot(xc, [yc[0], yc[0]], 'k-', label='cloud'); ax.plot(xc, [yc[1], yc[1]], 'k-')
ax.plot([xc[0], xc[0]], yc, 'k-'); ax.plot([xc[1], xc[1]], yc, 'k-')

# TODO: pick a vegetation region xv, yv (look for a greenish patch of land) and plot it in green,
# the same way xl/yl and xc/yc are plotted above.

ax.legend()
plt.show()


## Task 2 [10 pts] -- Color relationships between scene types

Now we sub-set the actual pixel values (not just the picture) for each region, and look at how
the channels relate to each other -- the same kind of channel-vs-channel scatter we did earlier
in the course.

**TODO:** sub-set the red/green/blue values for your vegetation box (following the `redl`/`grnl`/
`blul` and `redc`/`grnc`/`bluc` pattern below for ocean/cloud), then add vegetation to the scatter
plot. In a short markdown note below your plot: does the vegetation point cloud land where you'd
expect, given that healthy vegetation reflects strongly in green and absorbs red and blue?


In [ ]:
# --- Provided: ocean and cloud pixel subsets ---
redl = img[yl[0]:yl[1], xl[0]:xl[1], 0].flatten()
grnl = img[yl[0]:yl[1], xl[0]:xl[1], 1].flatten()
blul = img[yl[0]:yl[1], xl[0]:xl[1], 2].flatten()
redc = img[yc[0]:yc[1], xc[0]:xc[1], 0].flatten()
grnc = img[yc[0]:yc[1], xc[0]:xc[1], 1].flatten()
bluc = img[yc[0]:yc[1], xc[0]:xc[1], 2].flatten()

# TODO: pull redv/grnv/bluv out of your xv/yv box, the same way redl/grnl/blul etc. were pulled
# from xl/yl above (you'll need this same trio of arrays again in Task 3 and Task 5).


plt.plot(redl, grnl, '.', color='lightblue', label='ocean (grn)'); plt.plot(redl, blul, '.', color='blue', label='ocean (blu)')
plt.plot(redc, grnc, '.', color='gray', label='cloud (grn)'); plt.plot(redc, bluc, '.', color='black', label='cloud (blu)')
# TODO: add vegetation to this plot too -- two more plt.plot() calls (green vs. red, and blue vs.
# red), styled/labeled however you like, following the pattern of the ocean/cloud calls above.
plt.plot([0, 1], [0, 1], 'y--', label='1:1 line')
plt.xlabel('red'); plt.ylabel('grn & blu'); plt.title('Color relationships for three scene types')
plt.legend()
plt.show()


*Your answer here: does the vegetation point cloud's position make sense?*

## Task 3 [5 pts] -- A "spectrum" for each scene type

We only have 3 wavelengths here (red = 650 nm, green = 550 nm, blue = 440 nm), but we can still
plot "reflectance vs. wavelength" for a single pixel from each region -- a 3-point "spectrum."

**TODO:** add the vegetation spectrum (one pixel, `pxv = 0`) to the plot below, using the same
pattern as ocean and cloud.


In [ ]:
pxl = 0
plt.plot([650, 550, 440], [redl[pxl], grnl[pxl], blul[pxl]], 'bo-', label='ocean')
pxc = 0
plt.plot([650, 550, 440], [redc[pxc], grnc[pxc], bluc[pxc]], 'ko-', label='cloud')
# TODO: add a third "spectrum" line for vegetation, same pattern as ocean/cloud above (pick a
# pixel index into your redv/grnv/bluv arrays from Task 2).
plt.xlabel('wavelength [nm]'); plt.ylabel('Reflectance'); plt.ylim(0, 1); plt.legend()
plt.show()


## From spatial to spectral PCA

This is the `village.py` pipeline, unchanged, just with pixels as samples and (R, G, B) as the
3 features instead of houses and (EW, NS):

| | `village.py` (today, spatial) | this notebook (spectral) |
|---|---|---|
| sample | one house | one pixel |
| features | East-West, North-South | Red, Green, Blue (650, 550, 440 nm) |
| data array | `A`, shape `(n_houses, 2)` | `X`, shape `(n_pixels, 3)` |
| eigenvectors | 2 "self" directions in (EW, NS) space | 3 "self" directions in (R, G, B) space |

## Task 4 [10 pts]

The cell below builds `X`, standardizes it (or not -- see the comment), computes the mean and
covariance matrix, and does the eigenvalue decomposition -- read it line by line against the
`village.py` version from class, then answer:

a) What is the shape/format of `X` (the line that builds it)? Why do we `.flatten()` `red`,
   `grn`, `blu` first? <br>
b) `mean_vec` plays the same role as `center` did in `village.py`. What does it represent here --
   the mean of what, over what? <br>
c) In `village.py`, the eigenvalues told us how much variance was along the road vs. across it.
   What do the eigenvalues represent here?


In [ ]:
# --- Provided: build X, compute covariance matrix, eigen-decompose ---
red = img[:, :, 0].flatten()
grn = img[:, :, 1].flatten()
blu = img[:, :, 2].flatten()
n = len(red)
X = np.reshape(np.array([red, grn, blu], dtype=float), (3, n)).T   # shape (n_pixels, 3)

# Standardize? village.py left this as a toggle too -- we leave the data as-is for now.
X_std = X

mean_vec = np.mean(X_std, axis=0)
cov_mat = (X_std - mean_vec).T.dot(X_std - mean_vec) / (n - 1)
print('Mean (R, G, B):', mean_vec)

eig_vals, eig_vecs = np.linalg.eig(cov_mat)
# cov_mat is symmetric in theory, but floating-point round-off in the .dot() above can leave it
# *not quite* bit-for-bit symmetric -- numpy's general-purpose eig() then returns a complex dtype
# with ~0 imaginary parts (unlike village.py's 2x2 case, this bites you here). Taking .real is safe:
eig_vals = eig_vals.real
eig_vecs = eig_vecs.real
print('Eigenvalues:', eig_vals)
print('Eigenvectors (columns):')
print('   R:', eig_vecs[0])
print('   G:', eig_vecs[1])
print('   B:', eig_vecs[2])


*Your answers to a), b), c) here.*

## Scores: reprojecting data onto the eigenvectors [Task 5, 15 pts]

Recall from `village.py`: once we had the eigenvectors, we built `matrix_w` (the eigenvectors
stacked as columns) and reprojected the data with `A.dot(matrix_w)`. Each house ended up with two
new numbers -- its position along EV1 and along EV2. **Those numbers are the scores.** A score
isn't a direction (that's what the eigenvector/loading is) -- it's *how much of that direction* a
particular sample has.

Here, reprojecting gives every *pixel* a PC1 score and a PC2 score. Two things we can do with that:

1. Reproject just our three labeled regions (ocean/cloud/vegetation) and see if PCA space
   separates them better than raw (R, G, B) did.
2. Reproject **every pixel** in the image, then reshape the resulting scores back into the
   image's rows/columns -- turning a list of numbers back into a picture. That picture is a
   **scores map**: bright where that pattern is strong, dark where it's weak. This is exactly
   the kind of figure in the Shea, Pilewskie & Kindel (2011) slide from today ("these are the
   vegetation PC scores").

**TODO (a):** sort `eig_vals`/`eig_vecs` by eigenvalue magnitude (largest first) the same way
`village.py` did, and build `matrix_w` from the *top two* eigenvectors.

**TODO (b):** reproject your three regions' pixels (`X1`, `X2`, `X3` below) onto `matrix_w` and
scatter-plot them in PC1/PC2 space. Do ocean, cloud, and vegetation separate more cleanly here
than they did in raw (R, G, B) space back in Task 2?

**TODO (c):** reproject the *entire image* (`X`, already built above) onto `matrix_w`, reshape
each column back to `img.shape[0:2]`, and display PC1 and PC2 as images side by side with the
original. Which one looks like it's mostly separating "bright things from dark things" (clouds
vs. ocean/land)? Which one looks like it's picking out something more specific (e.g. vegetation)?


In [ ]:
# TODO (a): Build eig_pairs: a list of (abs(eigenvalue), eigenvector) tuples, one per
# eigenvector -- same idea as village.py's eig_pairs, just 3 entries instead of 2 (eig_vecs'
# COLUMNS are the eigenvectors, so eig_vecs[:, i] pairs with eig_vals[i]). Sort it so the
# LARGEST eigenvalue comes first. Then build matrix_w (shape 3x2) by stacking the top two
# eigenvectors as columns -- look at how village.py built its matrix_w from eig_pairs if you
# want a refresher, but don't just copy it verbatim: the shapes here are different (3 features,
# not 2), so make sure matrix_w actually comes out 3x2.


# TODO (b): Build X1, X2, X3 -- each an (n_pixels, 3) array -- by stacking your ocean/cloud/
# vegetation red*/grn*/blu* arrays column-wise (same idea as how X was built in Task 4, just
# per-region and without the .flatten() since these are already 1-D). Reproject each one onto
# matrix_w (matrix multiplication, not elementwise) to get Y1, Y2, Y3 -- each should come out
# (n_pixels, 2): a PC1 score and a PC2 score per pixel. Make ONE scatter plot with all three
# regions' scores on it (PC1 on the x-axis, PC2 on the y-axis), labeled and with a legend.


# TODO (c): Now reproject the FULL image's pixel array (the X you already built in Task 4, all
# ~1.25M pixels) onto matrix_w to get Y, shape (n_pixels, 2). Y[:, 0] and Y[:, 1] are both just
# long 1-D lists of numbers at this point -- reshape each one back to the image's original 2-D
# shape (img.shape[0:2]) to turn it into pc1_map and pc2_map. Display the original image,
# pc1_map, and pc2_map as three subplots side by side, each with a title.


## Task 6 [10 pts] -- Interpreting what you got

a) Look at your PC1 and PC2 score maps. In a sentence or two each: what does PC1 seem to be
   picking out spatially? What does PC2 seem to be picking out? <br>
b) The wavelengths are red = 650 nm, green = 550 nm, blue = 440 nm. Print `eig_vecs` for your
   top two eigenvectors and plot each one against wavelength (like we discussed briefly in
   class). Does either eigenvector's *shape* (as a function of wavelength) look like it matches
   a physically-recognizable spectrum (e.g. something that's higher in the green, like
   vegetation, or flatter/brighter across all three, like cloud)? Is it always possible to assign
   a clean physical meaning to a principal component? <br>
c) *(Optional, not graded)* If this image had 50 spectral channels instead of 3 (like a real
   hyperspectral imager), would you expect PCA to still work the same way? What would change?


*Your answers to a), b), (c) here.*

---

## Part B [15 pts] -- Project Progress #1: data access and a first plot

This is your first graded check-in on **Project Stage 1** (data access / visualization -- see
today's project slides). The bar is deliberately low this time around: we're checking that you
can *get your own data into Python and look at it*, not that you've done any analysis yet.

**The rule, restated from today's slides and from the email a few of you already got:** Panoply,
QGIS, and similar tools are great for *previewing* a dataset -- understanding its structure,
attributes, projection, etc. -- before you write code against it. They are **not** a substitute
for your own reader and plotting code. The actual data read and the actual plot in your
submission must be done in Python (`xarray`, `netCDF4`/`h5py`, `pandas`, `geopandas`, `rasterio`,
`cartopy`, `shapely`, etc. are all fine -- as long as *you* understand what they're doing). If you
used Panoply/QGIS to look at your data before writing code, that's expected and fine -- just don't
submit a screenshot from one of those tools in place of a Python-made plot.

**TODO:**

1. In 3-5 sentences (markdown cell below): what is your project dataset, where does it come from,
   and what are you hoping to learn from it (even if the plan is still rough -- that's fine, Stage
   2 is where the real analysis plan gets nailed down)?
2. In the code cell below: read your dataset with Python (whatever library is appropriate for its
   format), and produce **one** baseline plot or map -- a time series, a spatial map, a profile,
   whatever is the most natural "first look" at your data. It does not need to be polished or
   final.
3. If you're working in a group, you only need to submit this once per group -- just say so and
   list your group members.


*Your dataset description here (3-5 sentences): what it is, where it's from, what you're hoping to learn. If in a group, list members here.*

In [ ]:
# TODO: read your project dataset with Python and produce one baseline plot/map here.
# This is intentionally open-ended -- the format depends entirely on your dataset.


---
*Submit this notebook (with outputs run) either by pushing it to your GitHub repo, or as a
Jupyter notebook file upload on Canvas -- your choice. Questions: office hours or email.*